# Laboratory Exercise 1: Food Delivery

City Delivery is a delivery service that brings meals from restaurants across Macedonia to the door. Its pages are a stored snapshot in `data/site`, and the setup serves them from this machine, so every request stays on the local loopback address. The orders are synthetic and describe no real restaurant, courier, or customer.

Answer each request in the cells below it, and add as many cells as you need. The laboratory test asks for values and decisions that your notebook computes, so keep every result visible, run the notebook from top to bottom before you submit it, and make sure it runs on a machine without internet access.

## Setup

The cells in this section are written already. Run them before the tasks.

In [3]:
import json
import re
import threading
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import parse_qs, urljoin, urlparse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from bs4 import BeautifulSoup, Tag
from playwright.async_api import async_playwright
from scipy import stats
from statsmodels.stats.multitest import multipletests

DATA = Path("data")
# Every random step draws from this generator, so a rerun gives the same result.
rng = np.random.default_rng(2026)

In [4]:
# The course style.
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#1e2a24"]
GROUND, INK, GRID = "#ffffff", "#1e2a24", "#e2e6e4"
plt.rcParams.update(
    {
        "figure.figsize": (8, 4.5),
        "figure.facecolor": GROUND,
        "figure.constrained_layout.use": True,
        "savefig.facecolor": GROUND,
        "savefig.transparent": False,
        "font.family": "sans-serif",
        "font.sans-serif": ["Archivo", "DejaVu Sans"],
        "font.size": 12,
        "text.color": INK,
        "axes.facecolor": GROUND,
        "axes.edgecolor": INK,
        "axes.labelcolor": INK,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "axes.prop_cycle": plt.cycler(color=PALETTE),
        "grid.color": GRID,
        "xtick.color": INK,
        "ytick.color": INK,
        "legend.frameon": False,
        "image.cmap": "cividis",
    }
)
sns.set_palette(PALETTE)

In [5]:
# Every path is a file under the site directory.
class Service(SimpleHTTPRequestHandler):
    extensions_map = {
        **SimpleHTTPRequestHandler.extensions_map,
        ".html": "text/html; charset=utf-8",
    }

    def log_message(self, format: str, *args: object) -> None:
        pass


# Port 0 lets the system choose a free port, so a second running notebook never collides.
server = ThreadingHTTPServer(("127.0.0.1", 0), partial(Service, directory=DATA / "site"))
threading.Thread(target=server.serve_forever, daemon=True).start()
BASE = f"http://127.0.0.1:{server.server_port}/"
BASE

'http://127.0.0.1:40459/'

## Task 1: Collect the Orders

The service lists its delivered orders across several pages, and the first one is `catalogue/page-1.html` under `BASE`. Collect every order into a table named `orders`, with one row per order and these columns:

- `order_id`, the identifier of the order,
- `restaurant`, `cuisine`, and `city`,
- `distance` and `unit`, the distance from the restaurant to the door and the unit it is quoted in, exactly as the order states them,
- `minutes`, the minutes the delivery took,
- `slot`, `weather`, `payment`, and `rating`, the stars out of 5 that the customer gave, and
- `by_restaurant`, which is true when the restaurant delivered the order itself and false when a courier of the service did.

An order that does not state a field has a missing value in that column. Report how many orders the service lists and over how many pages they are spread.

In [6]:
response = requests.get(urljoin(BASE,"catalogue/page-1.html"))
soup = BeautifulSoup(response.text,"lxml")
soup.prettify()[:3000]


'<!DOCTYPE html>\n<html lang="en">\n <head>\n  <meta charset="utf-8"/>\n  <meta content="width=device-width, initial-scale=1" name="viewport"/>\n  <title>\n   Delivered orders, page 1 | City Delivery\n  </title>\n  <link href="../assets/site.css" rel="stylesheet"/>\n </head>\n <body>\n  <header class="site-header">\n   <div class="wrap">\n    <a class="brand" href="../index.html">\n     City Delivery\n    </a>\n    <nav aria-label="Main" class="primary-nav">\n     <a href="../index.html">\n      Home\n     </a>\n     <a aria-current="page" href="../catalogue/page-1.html">\n      Browse\n     </a>\n     <a href="#">\n      Help\n     </a>\n    </nav>\n   </div>\n  </header>\n  <main>\n   <div class="wrap">\n    <h1>\n     Delivered orders\n    </h1>\n    <div class="layout">\n     <section aria-label="Results">\n      <p class="toolbar">\n       192 results, showing 1 to 12\n      </p>\n      <div class="results">\n       <article class="card order" data-id="310069" id="order-310069">\n

In [7]:
COLUMNS = [
    "order_id", "restaurant", "cuisine", "city",
    "distance", "unit", "minutes", "slot",
    "weather", "payment", "rating", "by_restaurant",
]
rows = []
page_url = urljoin(BASE,"catalogue/page-1.html")
visited = set()
pages_count = 0

while page_url and page_url not in visited:
    visited.add(page_url)
    response = requests.get(page_url,timeout = 5)
    if response.status_code != 200:
        break
    pages_count += 1
    soup = BeautifulSoup(response.text,"lxml")
    order_cards = soup.select("article.card.order")

    for order in order_cards:
        row = {column: None for column in COLUMNS}
        row["order_id"] = order.get("data-id")

        restaurant = order.select_one("h2.title a, h2.title")
        if restaurant:
            row["restaurant"] = restaurant.getText(strip = True)
            
        for col in ["cuisine", "city", "slot", "weather", "payment", "rating"]:
            element = order.select_one(f".{col}")
            if element:
                value = element.get_text(" ", strip=True)
                if value not in ("", "-", "—"):
                    row[col] = value
        delivery = order.select_one(".delivery.figure, .delivery")
        if delivery:
            match = re.search(r"(\d+)\s*minutes?", delivery.get_text(" ", strip=True))
            if match:
                row["minutes"] = int(match.group(1))
        summary = order.select_one(".summary")
        if summary:
            match = re.search(r"([\d.]+)\s*(km|mi|m)\b", summary.get_text(" ", strip=True), re.IGNORECASE)
            if match:
                row["distance"] = float(match.group(1))
                row["unit"] = match.group(2)

        courier = order.select_one(".courier.muted, .courier")
        if courier:
            courier_text = courier.get_text(" ", strip=True).lower()
            if "restaurant" in courier_text:
                row["by_restaurant"] = True
            elif "courier" in courier_text or "service" in courier_text or "city delivery" in courier_text:
                row["by_restaurant"] = False

        if row["rating"] is not None:
            match = re.search(r"(\d+(?:\.\d+)?)", str(row["rating"]))
            if match:
                row["rating"] = float(match.group(1))

        rows.append(row)
    next_link = (
        soup.select_one("a[rel='next']")
        or soup.select_one("a.next")
        or soup.select_one("a[aria-label='Next']")
    )
    if next_link and next_link.get("href") and not next_link["href"].startswith("#"):
        page_url = urljoin(page_url, next_link["href"])
    else:
        page_url = None
orders = pd.DataFrame(rows, columns=COLUMNS)
print("Total orders:", len(orders))
print("Total pages:", pages_count)
print(orders.head())



Total orders: 192
Total pages: 16
  order_id    restaurant       cuisine    city  distance unit  minutes  \
0   310069    Salad Bowl         Salad  Skopje       5.0   km     23.0   
1   283894   Pizza House         Pizza  Skopje     650.0    m     30.0   
2   832866     Sushi Bar         Sushi  Skopje     700.0    m      NaN   
3   582431  Home Kitchen   Traditional  Skopje       2.7   km     25.0   
4   662069  Snack Corner  Small plates  Skopje       2.4   km     26.0   

         slot weather payment  rating  by_restaurant  
0  Late night   Clear    Cash     5.0          False  
1       Lunch   Clear    Card     3.0          False  
2       Lunch    Rain    Cash     3.0           True  
3       Lunch    Rain    Card     4.0          False  
4  Late night   Clear    Card     4.0           True  


## Task 2: Add the Receipts

The home page of the service, `index.html` under `BASE`, points to a page of receipts. Collect every receipt the page holds, and add the items, the total, and the tip of each order to `orders`, so that every order appears exactly once. Name the result `df`.

Report how many receipts the page says it holds, how many you collected, and what you did about any receipt that would put an order in `df` more than once.

In [8]:
receipts_url = urljoin(BASE, "receipts.html")
response = requests.get(receipts_url)
response.raise_for_status()

soup = BeautifulSoup(response.text, "lxml")
meta = soup.select_one("#receipts-meta")
print("Receipts information", meta.get_text(" ", strip=True) if meta else "No receipts found")
match = re.search(
    r"const\s+RECEIPTS\s*=\s*(\[.*?\])\s*;",
    response.text,
    re.DOTALL
)
receipts_data = json.loads(match.group(1))
receipts = pd.DataFrame(receipts_data)
receipts.head()
print("Total receipts:", len(receipts))
print("Duplicates:",  receipts["order_id"].duplicated().sum())
receipts = receipts.drop_duplicates(subset="order_id", keep="first")
orders_unique = orders.drop_duplicates(subset="order_id", keep="first")
receipts["order_id"] = receipts["order_id"].astype(str)
df = orders_unique.merge(
    receipts,
    on="order_id",
    how="left",
    validate="one_to_one"
)
print("Total receipts:", len(orders_unique))

Receipts information 
Total receipts: 193
Duplicates: 1
Total receipts: 192


## Task 3: Describe the Orders

Answer each request below from `df`. Support each answer with the numbers that establish it, and with a figure wherever the request asks you to show something.

### 3.1 Column Types

Give every column of `df` a type that matches what it measures. The slots have a natural order, from lunch to late night, and the type of `slot` keeps it.

In [19]:
print(df.dtypes)

print(df["slot"].unique())

df["slot"] = df["slot"].str.strip()

slot_order = [
    "Lunch",
    "Dinner",
    "Late night"
]
df["slot"] = pd.Categorical(
    df["slot"],
    categories=slot_order,
    ordered=True
)

print(df["slot"].dtype)
print(df["slot"].cat.categories)
print(df["slot"].cat.ordered)


order_id              str
restaurant            str
cuisine               str
city                  str
distance          float64
unit                  str
minutes           float64
slot             category
weather               str
payment               str
rating            float64
by_restaurant        bool
items               int64
total               int64
tip                 int64
dtype: object
['Late night', 'Lunch', 'Dinner']
Categories (3, str): ['Lunch' < 'Dinner' < 'Late night']
category
Index(['Lunch', 'Dinner', 'Late night'], dtype='str')
True


### 3.2 What Is Missing

Count the missing values in each column, show where they fall, and find out whether the absence of a value depends on another column.

In [23]:
missing_counts = df.isna().sum().sort_values(ascending=False)
print("Missing values per column:")
print(missing_counts.to_frame(name="missing_count"))
missing_rows = df[df.isna().any(axis=1)]
print("Num of rows with missing values", len(missing_rows))

print(missing_rows)


Missing values per column:
               missing_count
minutes                   42
rating                    24
restaurant                 0
cuisine                    0
city                       0
distance                   0
order_id                   0
unit                       0
slot                       0
weather                    0
payment                    0
by_restaurant              0
items                      0
total                      0
tip                        0
Num of rows with missing values 60
    order_id    restaurant       cuisine      city  distance unit  minutes  \
2     832866     Sushi Bar         Sushi    Skopje     700.0    m      NaN   
10    794180  Snack Corner  Small plates    Skopje       1.4   km      NaN   
17    312377  Burger Bar 7       Burgers    Tetovo       2.3   km     33.0   
23    421364    City Grill         Grill    Bitola       6.4   km      NaN   
28    751240   Pizza House         Pizza    Tetovo       2.2   km     19.0   
29    

### 3.3 Distances in Two Units

Some orders quote the distance in metres and others in kilometres. Compare the quoted distances within each unit, then add a column `distance_km` that states every distance in kilometres.

In [25]:
distance_summary = (
    df.groupby("unit")["distance"]
      .describe()
)

print(distance_summary)


      count        mean         std    min    25%    50%    75%    max
unit                                                                  
km    162.0    3.230247    1.276013    1.3    2.3    2.9    4.1    8.0
m      30.0  531.666667  201.067553  150.0  362.5  550.0  687.5  900.0


### 3.4 The Delivery Time

Describe the delivery time by its centre, its spread, and the shape of its distribution, and show the distribution. Decide which summary of the centre describes a typical order better.

In [10]:
# Write your solution here.

### 3.5 Delivery Time by Slot

Compare the delivery time across the slots.

In [11]:
# Write your solution here.

### 3.6 Orders by City

Show how the orders are spread across the cities, and give the share of orders in each city.

In [12]:
# Write your solution here.

### 3.7 Items and Total

Show how the number of items and the total of an order move together, and measure how strongly they do. Decide whether the relationship is close to a straight line.

In [13]:
# Write your solution here.

### 3.8 Every Pair of Numeric Columns

Show how strongly every pair of numeric columns moves together, measured by the Spearman correlation, and name the pair that moves together most strongly.

In [14]:
# Write your solution here.

### 3.9 Delivery Time by City and Slot

Find the combination of city and slot with the highest median delivery time, counting only the combinations with at least three orders that state a delivery time.

In [15]:
# Write your solution here.

## Task 4: Delivery Times

Use a significance level of 5 % and a confidence level of 95 % for every decision and every range in this task and the next. For each question, state the decision and give the range beside it. The questions of this task concern the orders that state a delivery time.

### 4.1 The Advertised Time

The home page states an average delivery time. Decide whether the average delivery time of the orders differs from that statement, and give a range for the average delivery time.

In [16]:
# Write your solution here.

### 4.2 Rain and Clear Weather

Decide whether the orders delivered in rain and in clear weather differ in their average delivery time, and give a range for the difference, taken as rain minus clear.

In [17]:
# Write your solution here.

## Task 5: Slots and Payments

The questions of this task concern all orders.

### 5.1 Payment by Slot

Decide whether the payment method depends on the slot of an order, and give a range for the share of cash payments in each slot, using a percentile bootstrap.

In [18]:
# Write your solution here.

### 5.2 The Mix of Slots

The home page states how the orders divide among the slots. Decide whether the orders match that statement, and give a range for the share of late-night orders, using a percentile bootstrap.

In [19]:
# Write your solution here.

In [20]:
server.shutdown()